# 02 - Pré-processing OpenAgenda

---

- **Projet 9 :** Concevez et déployez un système RAG
- **Auteur :** Justine Tranchant
- **Date :** mai 2026

---

Objectif : suivre le pipeline de pré-processing des événements OpenAgenda, depuis les données brutes jusqu'au dataset nettoyé.

Ce notebook reprend les mêmes fonctions que les scripts du projet, mais permet d'inspecter les données avec pandas à chaque étape.


## Imports et configuration

On importe les chemins du projet, les noms de fichiers configurés et les fonctions de pré-processing déjà utilisées par les scripts.


In [1]:
import sys
from datetime import datetime, timezone

import pandas as pd

# Permet d'importer src.* quand le notebook est exécuté depuis le dossier notebooks/.
sys.path.append("..")

from src.config import (
    OPENAGENDA_REFERENCE_DATE,
    PATHS,
    RAW_OPENAGENDA_EVENTS_FILENAME,
    PROCESSED_EVENTS_FILTERED_FILENAME,
    PROCESSED_EVENTS_CLEAN_FILENAME,
)
from src.preprocessing import (
    clean_events,
    filter_events_by_date,
    get_cleaning_stats,
    get_date_filtering_stats,
    save_clean_events,
    save_filtered_events,
)
from src.utils.io import load_json, load_json_to_df


In [2]:
raw_path = PATHS.data_raw / RAW_OPENAGENDA_EVENTS_FILENAME
filtered_path = PATHS.data_processed / PROCESSED_EVENTS_FILTERED_FILENAME
clean_path = PATHS.data_processed / PROCESSED_EVENTS_CLEAN_FILENAME

print(f"Données brutes : data/raw/{raw_path.name}")
print(f"Données filtrées : data/processed/{filtered_path.name}")
print(f"Données nettoyées : data/processed/{clean_path.name}")


Données brutes : data/raw/openagenda_events_raw.json
Données filtrées : data/processed/events_filtered.json
Données nettoyées : data/processed/events_clean.json


In [3]:
# Helper centralisé dans src.utils.io


## Chargement des événements bruts

Le fichier brut est généré par la commande :

```bash
poetry run python scripts/01_fetch_openagenda_events.py
```


In [4]:
if not raw_path.exists():
    raise FileNotFoundError(
        "Fichier brut introuvable. Lancez d'abord : "
        "poetry run python scripts/01_fetch_openagenda_events.py"
    )

raw_events = load_json(raw_path)

print(f"Nombre d'événements bruts : {len(raw_events)}")


Nombre d'événements bruts : 138


## Visualisation des données brutes

On transforme la liste d'événements bruts en DataFrame pour inspecter les colonnes disponibles et les premières lignes.


In [5]:
df_raw = load_json_to_df(raw_path, make_view=True)
df_raw

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,13708573,initiation-a-lastronomie-a-lanton,https://openagenda.com/econature/events/initia...,Initiation à l'astronomie à Lanton,Initiation à l'astronomie et observation aux i...,<p>✨ Découvrez le ciel nocturne comme vous ne ...,Tarifs : Enfant -18 ans 10€ ; Adulte 18 ans et...,"[""Nature"", ""Sortie nature"", ""astronomie""]",https://cdn.openagenda.com/main/7d4acecf130c4c...,NaN,...,99155778,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://www.eco-na...","[{""link"": ""https://www.eco-nature.org/experien..."
1,10772673,mai-a-velo-2026-1779336,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=209855""]",https://cdn.openagenda.com/main/9358618cf0ab46...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
2,49861420,mai-a-velo-2026-436875,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=202572""]",https://cdn.openagenda.com/main/bcca417c031c44...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
3,97562973,mai-a-velo-2026-23977,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=211745""]",https://cdn.openagenda.com/main/8ff0ec673d324a...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
4,21183348,mai-a-velo-2026-8312624,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=213084""]",https://cdn.openagenda.com/main/7f2f220d9af143...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,2431673,la-contrainte-de-sante-dans-les-demarches-1220191,https://openagenda.com/francetravail/events/la...,La contrainte de santé dans les démarches,Vous avez une contrainte de santé !,<p>Vous avez une contrainte de santé ! Cette r...,NaN,"[""S'informer""]",https://cdn.openagenda.com/main/0574cad4ec854c...,NaN,...,38495884,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
134,47343435,atelier-creation-dentreprise-avec-la-chambre-d...,https://openagenda.com/francetravail/events/at...,ATELIER CREATION D'ENTREPRISE AVEC LA CHAMBRE ...,Atelier CREATION D'ENTREPRISE Atelier réalisé ...,<p>Atelier CREATION D'ENTREPRISE Atelier réali...,NaN,"[""Création d'entreprise""]",https://cdn.openagenda.com/main/40678c08cb934b...,NaN,...,38495884,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
135,47971167,rendez-vous-individuels-avec-la-chambre-de-com...,https://openagenda.com/francetravail/events/re...,Rendez vous individuels avec la Chambre de Com...,Vous avez déjà engagé des démarches dans le ca...,<p>Vous avez déjà engagé des démarches dans le...,NaN,"[""Création d'entreprise""]",https://cdn.openagenda.com/main/39dffd5431a14b...,NaN,...,38495884,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
136,14686204,reunion-information-aeronautique-7103629,https://openagenda.com/francetravail/events/re...,REUNION INFORMATION AERONAUTIQUE,"Les métiers d'ajusteur monteur, intégrateur ca...","<p>Les métiers 

Les keywords "challenge-id=" seront supprimés car ils n’apportent pas d’information sémantique utile au RAG.

In [6]:
df_raw[
    df_raw["keywords_fr"]
    .fillna("")
    .astype(str)
    .str.contains("challenge", case=False, na=False)
]

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
1,10772673,mai-a-velo-2026-1779336,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=209855""]",https://cdn.openagenda.com/main/9358618cf0ab46...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
2,49861420,mai-a-velo-2026-436875,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=202572""]",https://cdn.openagenda.com/main/bcca417c031c44...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
3,97562973,mai-a-velo-2026-23977,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=211745""]",https://cdn.openagenda.com/main/8ff0ec673d324a...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
4,21183348,mai-a-velo-2026-8312624,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,"[""challenge-id=213084""]",https://cdn.openagenda.com/main/7f2f220d9af143...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN


### Champs manquants dans les données brutes

On regarde rapidement les champs les plus souvent absents.


In [7]:
missing_raw = df_raw.isna().mean().sort_values(ascending=False)
missing_raw

location_description_fr        1.000000
contributor_email              1.000000
location_phone                 1.000000
location_website               1.000000
category                       1.000000
location_links                 1.000000
onlineaccesslink               1.000000
contributor_organization       1.000000
location_access_fr             1.000000
contributor_contactposition    1.000000
contributor_contactname        1.000000
contributor_contactnumber      1.000000
accessibility_label_fr         0.992754
age_max                        0.992754
accessibility                  0.992754
location_district              0.992754
age_min                        0.992754
location_tags                  0.985507
location_image                 0.985507
location_imagecredits          0.985507
conditions_fr                  0.956522
imagecredits                   0.934783
links                          0.934783
location_insee                 0.920290
keywords_fr                    0.036232


## Filtrage temporel

Le POC utilise une date de référence figée afin de rendre le dataset reproductible.

Les événements conservés sont ceux dont la date de fin est supérieure ou égale à la date de référence.


In [8]:
reference_datetime = datetime.fromisoformat(OPENAGENDA_REFERENCE_DATE).replace(
    tzinfo=timezone.utc
)

reference_datetime

datetime.datetime(2026, 5, 1, 0, 0, tzinfo=datetime.timezone.utc)

In [9]:
filtered_events = filter_events_by_date(raw_events, reference_datetime)
date_filtering_stats = get_date_filtering_stats(raw_events, filtered_events)

date_filtering_stats


{'total_events': 138,
 'kept_events': 138,
 'excluded_events': 0,
 'excluded_ratio': 0.0}

### Sauvegarde du fichier filtré

Cette cellule reprend le comportement du script :

```bash
poetry run python scripts/02_filter_openagenda_events.py
```


In [10]:
save_filtered_events(filtered_events, filtered_path)

print(f"Fichier sauvegardé : data/processed/{filtered_path.name}")


Fichier sauvegardé : data/processed/events_filtered.json


## Visualisation des données filtrées

On inspecte le DataFrame après filtrage temporel.


In [11]:
df_filtered = pd.DataFrame(filtered_events)
df_filtered.shape

(138, 56)

In [12]:
df_filtered[
    [
        "uid",
        "title_fr",
        "location_city",
        "firstdate_begin",
        "lastdate_end",
        "canonicalurl",
    ]
]

,uid,title_fr,location_city,firstdate_begin,lastdate_end,canonicalurl
0,13708573,Initiation à l'astronomie à Lanton,Lanton,2026-01-05T19:30:00+00:00,2027-01-03T21:30:00+00:00,https://openagenda.com/econature/events/initia...
1,10772673,Mai à Vélo 2026,Mios,2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,https://openagenda.com/maiavelo/events/mai-a-v...
2,49861420,Mai à Vélo 2026,La Teste-de-Buch,2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,https://openagenda.com/maiavelo/events/mai-a-v...
3,97562973,Mai à Vélo 2026,Andernos-les-Bains,2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,https://openagenda.com/maiavelo/events/mai-a-v...
4,21183348,Mai à Vélo 2026,Andernos-les-Bains,2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,https://openagenda.com/maiavelo/events/mai-a-v...
...,...,...,...,...,...,...
133,2431673,La contrainte de santé dans les démarches,Andernos-les-Bains,2026-10-12T13:00:00+00:00,2026-10-12T15:00:00+00:00,https://openagenda.com/francetravail/events/la...
134,47343435,ATELIER CREATION D'ENTREPRISE AVEC LA CHAMBRE ...,Andernos-les-Bains,2026-11-16T08:30:00+00:00,2026-11-16T11:00:00+00:00,https://openagenda.com/francetravail/events/at...
135,47971167,Rendez vous individuels avec la Chambre de Com...,Andernos-les-Bains,2026-11-16T12:30:00+00:00,2026-11-16T13:15:00+00:00,https://openagenda.com/francetravail/events/re...
136,14686204,REUNION INFORMATION AERONAUTIQUE,Andernos-les-Bains,2026-11-19T08:00:00+00:00,2026-11-19T11:00:00+00:00,https://openagenda.com/francetravail/events/re...


In [13]:
pd.to_datetime(df_filtered["lastdate_end"], errors="coerce").agg(["min", "max"])

min   2026-05-05 10:30:00+00:00
max   2027-01-03 21:30:00+00:00
Name: lastdate_end, dtype: datetime64[us, UTC]

## Nettoyage et normalisation

On nettoie maintenant les événements filtrés pour obtenir un format homogène, plus simple à utiliser dans la suite du pipeline RAG.


In [14]:
cleaned_events = clean_events(filtered_events)
cleaning_stats = get_cleaning_stats(filtered_events, cleaned_events)

cleaning_stats

{'total_raw_events': 138,
 'total_clean_events': 138,
 'removed_events': 0,
 'duplicate_or_invalid_events': 0,
 'missing_description_count': 0,
 'missing_location_count': 0,
 'missing_coordinates_count': 0,
 'missing_url_count': 0,
 'missing_image_count': 0}

### Sauvegarde du fichier nettoyé

Cette cellule reprend le comportement du script :

```bash
poetry run python scripts/03_clean_openagenda_events.py
```


In [15]:
save_clean_events(cleaned_events, clean_path)

print(f"Fichier sauvegardé : data/processed/{clean_path.name}")


Fichier sauvegardé : data/processed/events_clean.json


## Visualisation des données nettoyées

Le DataFrame nettoyé contient uniquement les champs utiles et normalisés pour la suite du projet.


In [16]:
df_clean = pd.DataFrame(cleaned_events)
df_clean.shape

(138, 17)

In [17]:
df_clean.columns

Index(['event_id', 'title', 'description', 'conditions', 'city',
       'location_name', 'address', 'start_date', 'end_date', 'keywords', 'url',
       'image_url', 'latitude', 'longitude', 'source', 'attendance_mode',
       'status'],
      dtype='str')

In [18]:
df_clean

,event_id,title,description,conditions,city,location_name,address,start_date,end_date,keywords,url,image_url,latitude,longitude,source,attendance_mode,status
0,13708573,Initiation à l'astronomie à Lanton,Initiation à l'astronomie et observation aux i...,Tarifs : Enfant -18 ans 10€ ; Adulte 18 ans et...,Lanton,"Blagon, 33138 Lanton, France","Blagon, 33138 Lanton, France",2026-01-05T19:30:00+00:00,2027-01-03T21:30:00+00:00,"Nature, Sortie nature, astronomie",https://openagenda.com/econature/events/initia...,https://cdn.openagenda.com/main/7d4acecf130c4c...,44.783281,-0.934394,EcoNature,Sur place,Programmé
1,10772673,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,Mios,mios,"3 Avenue de la République, 33380 Mios, France",2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,,https://openagenda.com/maiavelo/events/mai-a-v...,https://cdn.openagenda.com/main/9358618cf0ab46...,44.605771,-0.938355,Challenges Geovelo,Sur place,Programmé
2,49861420,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,La Teste-de-Buch,DECATHLON LA TESTE DE BUCH #183,"1 Rue Georges Charpak, 33260 La Teste-de-Buch,...",2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,,https://openagenda.com/maiavelo/events/mai-a-v...,https://cdn.openagenda.com/main/bcca417c031c44...,44.616267,-1.127944,Challenges Geovelo,Sur place,Programmé
3,97562973,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,Andernos-les-Bains,Mairie Andernos Les Bains,"179 Boulevard de la République, 33510 Andernos...",2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,,https://openagenda.com/maiavelo/events/mai-a-v...,https://cdn.openagenda.com/main/8ff0ec673d324a...,44.743788,-1.101869,Challenges Geovelo,Sur place,Programmé
4,21183348,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,Andernos-les-Bains,COBAN,"6 Avenue de l'Espérance, 33510 Andernos-les-Ba...",2026-04-30T22:00:00+00:00,2026-05-31T21:59:59+00:00,,https://openagenda.com/maiavelo/events/mai-a-v...,https://cdn.openagenda.com/main/7f2f220d9af143...,44.745758,-1.087861,Challenges Geovelo,Sur place,Programmé
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,2431673,La contrainte de santé dans les démarches,Vous avez une contrainte de santé ! Cette réun...,NaN,Andernos-les-Bains,Andernos-les-Bains - Agence ANDERNOS,33510 Andernos-les-Bains,2026-10-12T13:00:00+00:00,2026-10-12T15:00:00+00:00,S'informer,https://openagenda.com/francetravail/events/la...,https://cdn.openagenda.com/main/0574cad4ec854c...,44.743736,-1.100045,Mes événements France Travail,Sur place,Programmé
134,47343435,ATELIER CREATION D'ENTREPRISE AVEC LA CHAMBRE ...,Atelier CREATION D'ENTREPRISE Atelier réalisé ...,NaN,Andernos-les-Bains,Andernos-les-Bains - Agence ANDERNOS,33510 Andernos-les-Bains,2026-11-16T08:30:00+00:00,2026-11-16T11:00:00+00:00,Création d'entreprise,https://openagenda.com/francetravail/events/at...,https://cdn.openagenda.com/main/40678c08cb934b...,44.743736,-1.100045,Mes événements France Travail,Sur place,Programmé
135,47971167,Rendez vous individuels avec la Chambre de Com...,Vous avez déjà engagé des démarches dans le ca...,NaN,Andernos-les-Bains,Andernos-les-Bains - Agence ANDERNOS,33510 Andernos-les-Bains,2026-11-16T12:30:00+00:00,2026-11-16T13:15:00+00:00,Création d'entreprise,https://openagenda.com/francetravail/events/re...,https://cdn.openagenda.com/main/39dffd5431a14b...,44.743736,-1.100045,Mes événements France Travail,Sur place,Programmé
136,14686204,REUNION INFORMATION AERONAUTIQUE,"Les métiers d'ajusteur monteur, intégrateur ca...",NaN,Andernos-les-Bains,Andernos-les-Bains - Agence ANDERNOS,33510 Andernos-les-Bains,2026-11-19T08:00:00+00:00,2026-11-19T11:00:00+00:00,"#TousMobilisés, Recrutement, S'informer",https://openagenda.com/francetravail/events/re...,https://cdn.openagenda.com/main/2b348de803b247...,44.743736,-1.100045,Mes événements France Travail,Sur place,Programmé


### Champs manquants après nettoyage

Les valeurs `None` sont conservées pour représenter proprement les informations absentes dans les données structurées.


In [19]:
missing_clean = df_clean.isna().mean().sort_values(ascending=False)
missing_clean


conditions         0.956522
event_id           0.000000
keywords           0.000000
attendance_mode    0.000000
source             0.000000
longitude          0.000000
latitude           0.000000
image_url          0.000000
url                0.000000
end_date           0.000000
title              0.000000
start_date         0.000000
address            0.000000
location_name      0.000000
city               0.000000
description        0.000000
status             0.000000
dtype: float64

### Aperçu des descriptions nettoyées

On vérifie que les descriptions ne contiennent plus de HTML gênant et qu'elles restent lisibles.


In [20]:
df_clean[["title", "description", "conditions", "keywords"]].head()


,title,description,conditions,keywords
0,Initiation à l'astronomie à Lanton,Initiation à l'astronomie et observation aux i...,Tarifs : Enfant -18 ans 10€ ; Adulte 18 ans et...,"Nature, Sortie nature, astronomie"
1,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,
2,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,
3,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,
4,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,


### Répartition par ville

Cette vue permet de vérifier que le périmètre géographique reste cohérent.


In [21]:
df_clean["city"].value_counts()


city
Andernos-les-Bains    79
La Teste-de-Buch      46
Biganos                3
Arès                   3
Audenge                2
Lanton                 1
Mios                   1
Le Teich               1
Salles                 1
Belin-Béliet           1
Name: count, dtype: int64

## Synthèse

- Les événements bruts sont chargés depuis `data/raw/openagenda_events_raw.json`.
- Le filtrage temporel utilise la date de référence figée du POC.
- Les événements filtrés sont sauvegardés dans `data/processed/events_filtered.json`.
- Les événements nettoyés sont sauvegardés dans `data/processed/events_clean.json`.
- Les champs textuels sont nettoyés du HTML.
- Les dates, coordonnées, liens, images et métadonnées utiles sont normalisés.
- Le dataset nettoyé est prêt pour l'étape suivante : construire les textes indexables pour le RAG.
